# 8.9 — GRU

A GRU keeps the spirit of gated memory but folds it into one hidden state, blending old belief with a proposed update.

This lesson is the lean gated recurrent cell: fewer gates than an LSTM, but the same instinct to protect useful history. Convex interpolation explains the update gate, while reset filtering decides how much previous state the candidate may inspect.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np

SEED = 87
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def softmax(scores, axis=-1):
    shifted = scores - np.max(scores, axis=axis, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=axis, keepdims=True)


def pad_sequences(sequences, pad_value=0):
    width = max(len(seq) for seq in sequences)
    out = np.full((len(sequences), width), pad_value, dtype=int)
    mask = np.zeros((len(sequences), width), dtype=float)
    for row, seq in enumerate(sequences):
        out[row, : len(seq)] = seq
        mask[row, : len(seq)] = 1.0
    return out, mask


def summarize_rung(rung):
    lengths = [len(seq) for seq in rung["sequences"]]
    vocab = sorted({token for seq in rung["sequences"] for token in seq})
    return {
        "name": rung["name"],
        "n": len(rung["sequences"]),
        "min_len": min(lengths),
        "max_len": max(lengths),
        "vocab": vocab,
    }


def build_sequence_classification_ladder(kind):
    base = [
        {"name": "D1 lesson toy", "sequences": [[1, 0, 1, 1]], "labels": [1], "dependency": 4},
        {"name": "D2 clean patterns", "sequences": [[1, 0, 1], [0, 0, 1], [1, 1, 0], [0, 1, 0], [1, 0, 0]], "labels": [1, 0, 1, 0, 0], "dependency": 3},
        {"name": "D3 long gaps and noise", "sequences": [[1, 0, 0, 1], [1, 2, 0, 0], [0, 2, 2, 1], [0, 0, 0, 0], [1, 0, 2, 1], [0, 1, 2, 0]], "labels": [1, 0, 0, 0, 1, 0], "dependency": 4},
        {"name": "D4 click/dialogue snippets", "sequences": [[3, 1, 0, 4], [2, 0, 4], [3, 2, 2, 4], [1, 1, 0, 0], [3, 0, 1, 4], [2, 2, 0, 4]], "labels": [1, 0, 1, 0, 1, 0], "dependency": 5},
        {"name": "D5 delayed dependency", "sequences": [[1, 0, 0, 0, 0, 4], [0, 0, 1, 0, 0, 4], [1, 2, 2, 2, 2, 0], [0, 2, 2, 2, 2, 4], [1, 0, 2, 0, 2, 4], [0, 0, 0, 0, 0, 0]], "labels": [1, 0, 0, 0, 1, 0], "dependency": 6},
    ]
    if kind == "lstm":
        base[0] = {"name": "D1 lesson gate toy", "sequences": [[2, 0, 2]], "labels": [1], "dependency": 3}
    if kind == "gru":
        base[0] = {"name": "D1 lesson interpolation toy", "sequences": [[2, 1, 0, 1]], "labels": [1], "dependency": 4}
    return base


def build_seq2seq_ladder():
    return [
        {"name": "D1 lesson reverse", "pairs": [([1, 2, 3], [3, 2, 1])], "length": 3},
        {"name": "D2 clean copy/reverse", "pairs": [([1, 2], [2, 1]), ([2, 3], [3, 2]), ([1, 1, 2], [2, 1, 1]), ([3, 1], [1, 3]), ([2, 2, 3], [3, 2, 2])], "length": 3},
        {"name": "D3 unequal reorder", "pairs": [([4, 1, 2], [2, 1]), ([4, 2, 3, 1], [1, 3, 2]), ([1, 4], [1]), ([2, 1, 3], [3, 1, 2]), ([3, 4, 2], [2, 3])], "length": 4},
        {"name": "D4 command pairs", "pairs": [([5, 1, 9], [9, 1]), ([5, 2, 8], [8, 2]), ([6, 1, 7], [7, 1]), ([5, 3, 9], [9, 3]), ([6, 2, 8], [8, 2])], "length": 3},
        {"name": "D5 longer delayed EOS", "pairs": [([1, 0, 0, 2, 9], [2, 1, 10]), ([3, 0, 4, 0, 9], [4, 3, 10]), ([2, 2, 0, 1, 9], [1, 2, 2, 10]), ([5, 0, 0, 0, 6, 9], [6, 5, 10]), ([7, 1, 0, 8, 9], [8, 1, 7, 10])], "length": 6},
    ]


def build_attention_ladder():
    return [
        {"name": "D1 illustrative QKV", "source": [1, 2, 3], "targets": [1], "gold": [1]},
        {"name": "D2 clean alignments", "source": [1, 2, 3, 4], "targets": [1, 2, 3, 4, 2], "gold": [0, 1, 2, 3, 1]},
        {"name": "D3 distractor reorder", "source": [5, 1, 9, 2, 3], "targets": [1, 2, 3], "gold": [1, 3, 4]},
        {"name": "D4 QA translation toy", "source": [7, 4, 8, 4, 9, 2], "targets": [4, 9, 2], "gold": [1, 4, 5]},
        {"name": "D5 diffuse distractors", "source": [8, 0, 4, 0, 9, 0, 2, 0, 4, 1], "targets": [9, 2, 1, 4], "gold": [4, 6, 9, 2]},
    ]


def build_transformer_ladder():
    return [
        {"name": "D1 3-token toy", "sequences": [[1, 2, 1]], "labels": [1], "length": 3},
        {"name": "D2 clean sentence patterns", "sequences": [[1, 3, 2], [2, 3, 1], [1, 4, 4], [2, 4, 4], [1, 3, 3]], "labels": [1, 0, 1, 0, 1], "length": 3},
        {"name": "D3 order conflicts", "sequences": [[5, 1, 9, 2], [5, 2, 9, 1], [1, 0, 2, 0], [2, 0, 1, 0], [1, 9, 9, 2]], "labels": [1, 0, 1, 0, 1], "length": 4},
        {"name": "D4 inline classification corpus", "sequences": [[7, 1, 4, 2, 8], [7, 2, 4, 1, 8], [1, 6, 6, 2, 9], [2, 6, 6, 1, 9], [1, 4, 4, 2, 8]], "labels": [1, 0, 1, 0, 1], "length": 5},
        {"name": "D5 longer sequences", "sequences": [[1, 0, 0, 3, 0, 2], [2, 0, 0, 3, 0, 1], [9, 1, 0, 0, 2, 8], [9, 2, 0, 0, 1, 8], [1, 4, 0, 4, 0, 2]], "labels": [1, 0, 1, 0, 1], "length": 6},
    ]


def accuracy(predictions, labels):
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    return float(np.mean(predictions == labels))


## The concept, built once (D1)

A GRU folds gated memory into one hidden state. This lesson uses the convention $$h_t=(1-z_t)h_{old}+z_t\tilde h_t$$, so larger $z_t$ moves more strongly toward the candidate.

In [ ]:

def gru_step(x_t, h_prev, z_t, r_t, weights=None):
    if weights is None:
        weights = {"W_x": 0.7, "W_h": 0.4}
    reset_state = r_t * h_prev
    candidate = math.tanh(weights["W_x"] * x_t + weights["W_h"] * reset_state)
    h_t = (1.0 - z_t) * h_prev + z_t * candidate
    return h_t, candidate, reset_state

interpolated = []
for z in [0.0, 0.25, 0.5, 0.75, 1.0]:
    h_t = (1.0 - z) * 2.0 + z * -1.0
    interpolated.append(h_t)
assert np.allclose(interpolated, [2.0, 1.25, 0.5, -0.25, -1.0])
interpolated


The reset gate acts before candidate construction, not on the final interpolation. With $x=1$, $h=2$, and $r=0.25$, the candidate uses $0.25\cdot2=0.5$ inside $\tanh(0.7\cdot1+0.4\cdot0.5)$.

In [ ]:

h_t, candidate, reset_state = gru_step(1.0, 2.0, z_t=1.0, r_t=0.25)
assert np.isclose(reset_state, 0.5)
assert np.isclose(candidate, 0.7162978701990245)
assert np.isclose(0.95 ** 30, 0.21463876394293727)
print("candidate", candidate)


## The dataset ladder

The ladder progresses from interpolation to selective sparse updates with reset-needed distractors.

In [ ]:

rungs = build_sequence_classification_ladder("gru")
for rung in rungs:
    print(summarize_rung(rung))
    print("sample", rung["sequences"][0], "label", rung["labels"][0])


## Run the same method across D1--D5

The same GRU cell handles all rungs. Informative tokens get larger update gates, while zeros mostly keep the previous state.

In [ ]:

def gru_sequence_classifier(seq, flipped=False):
    h = 0.0
    states = []
    gate_rows = []
    for token in seq:
        informative = 1.0 if token in [1, 3, 4] else 0.0
        negative = 1.0 if token == 2 else 0.0
        z = 0.8 if informative or negative else 0.05
        r = 0.2 if negative else 1.0
        if flipped:
            z_used = 1.0 - z
        else:
            z_used = z
        x_value = 1.0 if informative else -1.0 if negative else 0.0
        h, candidate, reset_state = gru_step(x_value, h, z_used, r)
        states.append(h)
        gate_rows.append([z_used, r])
    prediction = int(h > 0.25)
    return prediction, h, np.asarray(states), np.asarray(gate_rows)

results = []
for rung in rungs:
    preds = [gru_sequence_classifier(seq)[0] for seq in rung["sequences"]]
    results.append({"rung": rung["name"], "delay": rung["dependency"], "accuracy": accuracy(preds, rung["labels"])})

for row in results:
    print(f'{row["rung"]:30s} delay={row["delay"]:2d} accuracy={row["accuracy"]:.3f}')


## Results visualization

The panels show update/reset gate traces, then the accuracy curve across delay and noise.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(15, 5))
for index, rung in enumerate(rungs):
    _, _, states, gates = gru_sequence_classifier(rung["sequences"][0])
    heat = np.vstack([states, gates[:, 0], gates[:, 1]])
    axes[0, index].imshow(heat, aspect="auto", cmap="cividis")
    axes[0, index].set_title(rung["name"].split()[0])
    axes[0, index].set_yticks([0, 1, 2])
    axes[0, index].set_yticklabels(["h", "z", "r"])

axes[1, 0].plot([row["delay"] for row in results], [row["accuracy"] for row in results], marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_xlabel("delay/noise")
axes[1, 0].set_ylabel("accuracy")
axes[1, 0].set_title("metric curve")
for axis in axes[1, 1:]:
    axis.axis("off")
plt.tight_layout()


## Pitfall on D5: flipping the update-gate convention

Some libraries use the complement convention. If we interpret $z_t$ backward, the same gate heatmap says the opposite thing and D5 decisions change.

In [ ]:

d5 = rungs[-1]
wrong_preds = [gru_sequence_classifier(seq, flipped=True)[0] for seq in d5["sequences"]]
fixed_preds = [gru_sequence_classifier(seq, flipped=False)[0] for seq in d5["sequences"]]
print("wrong flipped-convention D5 accuracy", accuracy(wrong_preds, d5["labels"]))
print("fixed lesson-convention D5 accuracy", accuracy(fixed_preds, d5["labels"]))


## Evaluate it + Practice

- Metric: sequence classification accuracy.
- No-skill baseline: majority label for the rung.
- Cheap sanity check: interpolation asserts [2, 1.25, 0.5, -0.25, -1].
- Ablation: flip the update convention and compare D5.
- Failure signals: reset gate appears to change final interpolation directly.

Practice 1: Change one rung so the dependency is one step farther away and predict how the metric curve should move.

Practice 2: Set reset to zero on a distractor and inspect the candidate.

Practice 3: Replace sparse updates with constant update 0.5 and compare the metric curve.